# 1.2 - Feature engineering + split (`accidentes_clean.csv`)

**Input:** `accidentes_clean.csv` from `data_cleaning.ipynb`.

**Outputs (same folder):** `accidentes_train.csv`, `accidentes_test.csv`, `variable_dictionary.csv`.

## Decisions

**Target.** `grave` is the target only.

**New columns**

- `franja_horaria`: clock hour of `hora_decimal`. `noche` 22–5, `manana` 6–11, `tarde` 12–17, `tarde-noche` 18–21.
- `con_moto`: 1 if `motos_implicadas >= 1`.
- `con_pesado`: 1 if `pesados_implicados >= 1`.
- `carretera_grupo`: keep `M30`, `A2`, `M607`, `A5`, `A4`, `M11`, `A3`. Every other road, including blanks, is `Other`.

**Used as recorded**

- `tipo_colision`, `estado_firme`, `tipo_via`, `ubicacion`, `tipo_accidente`, `circulacion`, `distrito`
- `vehiculos_implicados`, `hora_decimal` (`hora_decimal` also feeds `franja_horaria`)

**Left out of the table**

- `heridos_leves`, `heridos_graves`, `victimas_mortales`: they define `grave`.
- `numero_incidencia`, `localizacion`, `eje`, `unidad_samur_id`, `descripcion`, `fecha`, `hora`, `fecha_calendario`
- `motos_implicadas`, `pesados_implicados`: replaced by the two flags.
- `carretera`: replaced by `carretera_grupo`.

**Split.** Features are built on the full clean table, then an 80/20 split stratified on `grave` (`random_state` 42), so both sides keep the ~2.2% grave rate.


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

CLEAN = Path("./data/accidentes_clean.csv")
OUT_DIR = CLEAN.parent

df = pd.read_csv(CLEAN)
print("loaded:", CLEAN.resolve())
print("shape:", df.shape)
print("grave rate:", f"{100 * df['grave'].mean():.2f}% ({df['grave'].sum()} / {len(df)})")
df.head(3)


loaded: /Users/lukas/dev/UC3M/Project-UC3M/1_2_severity/data/accidentes_clean.csv
shape: (1962, 24)
grave rate: 2.29% (45 / 1962)


,fecha,hora,numero_incidencia,localizacion,ubicacion,carretera,eje,tipo_via,tipo_accidente,tipo_colision,...,circulacion,heridos_leves,heridos_graves,victimas_mortales,descripcion,distrito,unidad_samur_id,fecha_calendario,hora_decimal,grave
0,01/01/2024 0:00,1:30:48,20240101-0001,08NL10,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,NaN,1.0,0,0,SEÑALIZAR ACCIDENTE POR ALCANCE EN CARRIL CENT...,CENTRO,49.0,2024-01-01,1.500000,0
1,01/01/2024 0:00,1:59:17,20240101-0002,07NC10,SUPERFICIE,M30,NC,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,"AVISO DE ALCANCE ENTRE 2 VEHÍCULOS, ANTES DE L...",USERA,NaN,2024-01-01,1.983333,0
2,01/01/2024 0:00,5:58:11,20240101-0007,03FT45,TÚNEL,A5,FT,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,ACCIDENTE LEVE ENTRE DOS VEHICULOS (ALCANCE) E...,VICALVARO,29.0,2024-01-01,5.966667,0


## Feature set

Never use as inputs: `heridos_leves`, `heridos_graves`, `victimas_mortales` (and the label `grave` is the target only).
Drop high-cardinality IDs for v1: `numero_incidencia`, `localizacion`, `eje`, `unidad_samur_id`, `descripcion`.


In [7]:
def franja_horaria(h):
    # hora_decimal is fractional clock time (14.75 = 14:45). Bin on the clock hour.
    # noche 22-5 | manana 6-11 | tarde 12-17 | tarde-noche 18-21
    if pd.isna(h):
        return pd.NA
    h = int(h)
    if h >= 22 or h <= 5:
        return "noche"
    if 6 <= h <= 11:
        return "manana"
    if 12 <= h <= 17:
        return "tarde"
    return "tarde-noche"
TOP_ROADS = {"M30", "A2", "M607", "A5", "A4", "M11", "A3"}

feat = df.copy()
feat["franja_horaria"] = feat["hora_decimal"].map(franja_horaria)
feat["con_moto"] = (feat["motos_implicadas"].fillna(0).astype(float) >= 1).astype(int)
feat["con_pesado"] = (feat["pesados_implicados"].fillna(0).astype(float) >= 1).astype(int)
feat["carretera_grupo"] = feat["carretera"].where(feat["carretera"].isin(TOP_ROADS), "Other")

FEATURE_CATS = [
    "tipo_colision", "estado_firme", "tipo_via", "ubicacion",
    "tipo_accidente", "circulacion", "franja_horaria", "carretera_grupo",
    "distrito",
]
FEATURE_NUMS = ["vehiculos_implicados", "con_moto", "con_pesado", "hora_decimal"]
TARGET = "grave"

keep = FEATURE_CATS + FEATURE_NUMS + [TARGET]
missing = [c for c in keep if c not in feat.columns]
assert not missing, missing

X_y = feat[keep].copy()
print("feature table:", X_y.shape)
X_y.head(3)


feature table: (1962, 14)


,tipo_colision,estado_firme,tipo_via,ubicacion,tipo_accidente,circulacion,franja_horaria,carretera_grupo,distrito,vehiculos_implicados,con_moto,con_pesado,hora_decimal,grave
0,COLISIÓN POR ALCANCE,Seco y limpio,VIA DE SERVICIO,SUPERFICIE,Colision entre vehiculos,NaN,noche,M30,CENTRO,4.0,0,0,1.500000,0
1,COLISIÓN POR ALCANCE,Seco y limpio,VIA DE SERVICIO,SUPERFICIE,Colision entre vehiculos,FLUIDA,noche,M30,USERA,2.0,0,0,1.983333,0
2,COLISIÓN POR ALCANCE,Seco y limpio,TRONCO,TÚNEL,Colision entre vehiculos,FLUIDA,noche,A5,VICALVARO,2.0,0,0,5.966667,0


## Checks on the engineered columns

Counts and the grave rate inside each new column. 

In [8]:
def tasa_por(frame, col):
    g = (
        frame.groupby(col, dropna=False)[TARGET]
        .agg(n="size", graves="sum")
        .sort_values("n", ascending=False)
    )
    g["tasa_grave_%"] = (100 * g["graves"] / g["n"]).round(2)
    return g

for col in ["franja_horaria", "con_moto", "con_pesado", "carretera_grupo"]:
    print(f"\n=== {col} ===")
    print(tasa_por(X_y, col).to_string())

print("\nhora_decimal range:", float(X_y["hora_decimal"].min()), "→", float(X_y["hora_decimal"].max()))
print("M607 rows:", int((X_y["carretera_grupo"] == "M607").sum()))


=== franja_horaria ===
                  n  graves  tasa_grave_%
franja_horaria                           
tarde           778      16          2.06
manana          534       8          1.50
tarde-noche     445       8          1.80
noche           205      13          6.34

=== con_moto ===
             n  graves  tasa_grave_%
con_moto                            
0         1591      16          1.01
1          371      29          7.82

=== con_pesado ===
               n  graves  tasa_grave_%
con_pesado                            
0           1806      40          2.21
1            156       5          3.21

=== carretera_grupo ===
                    n  graves  tasa_grave_%
carretera_grupo                            
M30              1625      39          2.40
A2                 97       2          2.06
M607               68       2          2.94
A5                 61       0          0.00
A4                 38       1          2.63
Other              30       1          3.33
M11  

## Stratified 80/20 split

Preserve the ~2.2% grave rate in both folds. Prefer `sklearn.model_selection.train_test_split`; pandas fallback if sklearn is missing.


In [9]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

def stratified_split(frame, target, test_size=0.2, random_state=42):
    try:
        from sklearn.model_selection import train_test_split
        return train_test_split(
            frame, test_size=test_size, random_state=random_state,
            stratify=frame[target],
        )
    except ImportError:
        rng = np.random.RandomState(random_state)
        parts = []
        for _, g in frame.groupby(target):
            idx = np.array(g.index.to_numpy(), copy=True)
            rng.shuffle(idx)
            n_test = max(1, int(round(len(idx) * test_size)))
            test_idx = idx[:n_test]
            train_idx = idx[n_test:]
            parts.append((train_idx, test_idx))
        train_ix = np.concatenate([p[0] for p in parts])
        test_ix = np.concatenate([p[1] for p in parts])
        rng.shuffle(train_ix)
        rng.shuffle(test_ix)
        return frame.loc[train_ix].copy(), frame.loc[test_ix].copy()

train_df, test_df = stratified_split(X_y, TARGET, TEST_SIZE, RANDOM_STATE)

def rate(s):
    return f"{100 * s.mean():.2f}% ({int(s.sum())}/{len(s)})"

print("train:", train_df.shape, "grave", rate(train_df[TARGET]))
print("test: ", test_df.shape, "grave", rate(test_df[TARGET]))

train_path = OUT_DIR / "accidentes_train.csv"
test_path = OUT_DIR / "accidentes_test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
print("wrote:", train_path.resolve())
print("wrote:", test_path.resolve())


train: (1569, 14) grave 2.29% (36/1569)
test:  (393, 14) grave 2.29% (9/393)
wrote: /Users/lukas/dev/UC3M/Project-UC3M/1_2_severity/data/accidentes_train.csv
wrote: /Users/lukas/dev/UC3M/Project-UC3M/1_2_severity/data/accidentes_test.csv


## Variable dictionary

Short dictionary for the meeting: name, role, type, notes.


In [10]:
dictionary = [
    {"variable": "grave", "role": "target", "type": "binary",
     "notes": "1 if heridos_graves>0 OR victimas_mortales>0; ~2.2% positive"},
    {"variable": "tipo_colision", "role": "feature", "type": "categorical",
     "notes": "Cleaned"},
    {"variable": "estado_firme", "role": "feature", "type": "categorical",
     "notes": "Cleaned"},
    {"variable": "tipo_via", "role": "feature", "type": "categorical",
     "notes": "Cleaned"},
    {"variable": "ubicacion", "role": "feature", "type": "categorical",
     "notes": "Superficie / Tunel"},
    {"variable": "tipo_accidente", "role": "feature", "type": "categorical",
     "notes": "Collision type family"},
    {"variable": "circulacion", "role": "feature", "type": "categorical",
     "notes": "Traffic density; cleaned in M1"},
    {"variable": "franja_horaria", "role": "feature", "type": "categorical",
     "notes": "From hora_decimal: noche 22-5, manana 6-11, tarde 12-17, tarde-noche 18-21"},
    {"variable": "carretera_grupo", "role": "feature", "type": "categorical",
     "notes": "Top roads after M1 cleaning (M30, A2, M607, A5, A4, M11, A3) + Other"},
    {"variable": "distrito", "role": "feature", "type": "categorical",
     "notes": "District; cleaned in M1; about 25 levels"},
    {"variable": "vehiculos_implicados", "role": "feature", "type": "numeric",
     "notes": "-1 treated as missing in M1"},
    {"variable": "con_moto", "role": "feature", "type": "binary",
     "notes": "1 if motos_implicadas >= 1"},
    {"variable": "con_pesado", "role": "feature", "type": "binary",
     "notes": "1 if pesados_implicados >= 1"},
    {"variable": "hora_decimal", "role": "feature", "type": "numeric",
     "notes": "Fractional clock time in [0, 24); feeds franja_horaria"},
    {"variable": "heridos_* / victimas_mortales", "role": "excluded_leakage", "type": "numeric",
     "notes": "Never as inputs"},
    {"variable": "descripcion", "role": "excluded", "type": "text",
     "notes": "Deferred to task 2.1 RAG"},
]

dict_df = pd.DataFrame(dictionary)
dict_path = OUT_DIR / "variable_dictionary.csv"
dict_df.to_csv(dict_path, index=False)
print("wrote:", dict_path.resolve())
dict_df


wrote: /Users/lukas/dev/UC3M/Project-UC3M/1_2_severity/data/variable_dictionary.csv


,variable,role,type,notes
0,grave,target,binary,1 if heridos_graves>0 OR victimas_mortales>0; ...
1,tipo_colision,feature,categorical,Cleaned
2,estado_firme,feature,categorical,Cleaned
3,tipo_via,feature,categorical,Cleaned
4,ubicacion,feature,categorical,Superficie / Tunel
5,tipo_accidente,feature,categorical,Collision type family
6,circulacion,feature,categorical,Traffic density; cleaned in M1
7,franja_horaria,feature,categorical,"From hora_decimal: noche 22-5, manana 6-11, ta..."
8,carretera_grupo,feature,categorical,"Top roads after M1 cleaning (M30, A2, M607, A5..."
9,distrito,feature,categorical,District; cleaned in M1; about 25 levels
